In [3]:
%pip install -r ../requirements.txt

  Using cached attrs-23.2.0-py3-none-any.whl.metadata (9.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.6/77.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.4/199.4 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.5/85.5 kB 11.3 MB/s eta 0:00:00
Using cached attrs-23.2.0-py3-none-any.whl (60 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 335.8/335.8 kB 17.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install pyro-ppl seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 745.2/745.2 kB 13.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 9.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [5]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import pandas as pd
import pyro
import pyro.distributions as dist
from pyro.infer import MCMC, NUTS
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from devinterp.optim.sgld import SGLD
from devinterp.optim.sgnht import SGNHT
from devinterp.slt import estimate_learning_coeff_with_summary, sample
from devinterp.slt.llc import LLCEstimator, SamplerCallback, OnlineLLCEstimator
from devinterp.utils import plot_trace

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
sns.set_style("whitegrid")

# plotting
CMAP = sns.color_palette("muted", as_cmap=True)
PRIMARY, SECONDARY, TERTIARY = CMAP[:3]

/var/folders/ds/qvg7h1s93x16vvczwp8vr83m0000gn/T/ipykernel_10577/2903749014.py:4: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd
/Users/ragharao/opt/anaconda3/envs/devinterp_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# constants
SIGMA = 0.25
NUM_TRAIN_SAMPLES = 1000
BATCH_SIZE = NUM_TRAIN_SAMPLES
CRITERION = F.mse_loss

In [7]:
class SingularModel(nn.Module):
    def __init__(self, powers):
        super(SingularModel, self).__init__()
        self.weights = nn.Parameter(
            torch.tensor([0.0, 0.0], dtype=torch.float32, requires_grad=True).to(DEVICE)
        )
        self.powers = powers

    def forward(self, x):
        multiplied = torch.prod(self.weights**self.powers)
        x = x * multiplied
        return x

def generate_dataset_for_seed(seed=0):
    torch.manual_seed(seed)
    np.random.seed(seed)
    x = torch.normal(0, 2, size=(NUM_TRAIN_SAMPLES,))
    y = SIGMA * torch.normal(0, 1, size=(NUM_TRAIN_SAMPLES,))
    train_data = TensorDataset(x, y)
    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
    return train_loader, train_data, x, y

In [12]:
train_loader, train_data, x, y = generate_dataset_for_seed(0)
lr = 0.0001
num_chains = 3
num_draws = 5000
powers_to_test = [
    torch.tensor([0, 2]).to(DEVICE), # minimally singular
    torch.tensor([2, 0]).to(DEVICE) # minimally sigular
]
rlct_estimates_sgld = {
    tuple(power.detach().cpu().tolist()): estimate_learning_coeff_with_summary(
        model=SingularModel(power),
        loader=train_loader,
        optimizer_kwargs=dict(
            lr=lr,
            bounding_box_size=1.0,
        ),
        criterion=CRITERION,
        sampling_method=SGLD,
        num_chains=num_chains,
        num_draws=num_draws,
        device=DEVICE,
    )
    for power in powers_to_test
}
rlct_estimates_sgnht = {
    tuple(power.detach().cpu().tolist()): estimate_learning_coeff_with_summary(
        model=SingularModel(power),
        loader=train_loader,
        optimizer_kwargs=dict(
            lr=lr,
            bounding_box_size=1.0,
            diffusion_factor=0.01,
        ),
        criterion=CRITERION,
        sampling_method=SGNHT,
        num_chains=num_chains,
        num_draws=num_draws,
        device=DEVICE,
    )
    for power in powers_to_test
}

/Users/ragharao/opt/anaconda3/envs/devinterp_env/lib/python3.11/site-packages/devinterp/slt/sampler.py:170: UserWarning: You are taking more draws than burn-in steps, your LLC estimates will likely be underestimates. Please check LLC chain convergence.
  warnings.warn(
/Users/ragharao/opt/anaconda3/envs/devinterp_env/lib/python3.11/site-packages/devinterp/slt/sampler.py:174: UserWarning: You are taking more sample batches than there are dataloader batches available, this removes some randomness from sampling but is probably fine. (All sample batches beyond the number dataloader batches are cycled from the start, f.e. 9 samples from [A, B, C] would be [B, A, C, B, A, C, B, A, C].)
  warnings.warn(
/Users/ragharao/opt/anaconda3/envs/devinterp_env/lib/python3.11/site-packages/devinterp/slt/sampler.py:58: UserWarning: You are taking more sample batches than there are dataloader batches available, this removes some randomness from sampling but is probably fine. (All sample batches beyond th

In [13]:
for sampler_type, estimates in zip(
    ("sgld", "sgnht"), (rlct_estimates_sgld, rlct_estimates_sgnht)
):
    df_data = {
        "powers": [k for k in estimates.keys()],
        "LLC_Summary": estimates.values(),
    }
    df = pd.DataFrame(df_data)
    df["llc_mean"] = df["LLC_Summary"].apply(lambda x: x["llc/mean"])
    df["llc_std"] = df["LLC_Summary"].apply(lambda x: x["llc/std"])
    df = df.drop("LLC_Summary", axis=1)
    print(sampler_type)
    print(df.to_string(index=False))

sgld
powers  llc_mean  llc_std
(0, 2)  0.235147 0.050897
(2, 0)  0.177827 0.032972
sgnht
powers  llc_mean  llc_std
(0, 2)  0.279457 0.024806
(2, 0)  0.281580 0.024980
